### Churn Prediction Model Evaluation

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#### 1.0 Data Preparation 

In [2]:
data = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv.xls")
df = data
df.head().T

,0,1,2,3,4
customerID,7590-VHVEG,5575-GNVDE,3668-QPYBK,7795-CFOCW,9237-HQITU
gender,Female,Male,Male,Male,Female
SeniorCitizen,0,0,0,0,0
Partner,Yes,No,No,No,No
Dependents,No,No,No,No,No
tenure,1,34,2,45,2
PhoneService,No,Yes,Yes,No,Yes
MultipleLines,No phone service,No,No,No phone service,No
InternetService,DSL,DSL,DSL,DSL,Fiber optic
OnlineSecurity,No,Yes,Yes,Yes,No


In [3]:
df.columns = df.columns.str.lower().str.replace(' ', '_')
cat_cols = list(df.dtypes[df.dtypes == 'object'].index)

for i in cat_cols:
    df[i] = df[i].str.lower().str.replace(' ', '_')   # Ensuring uniformity in data columns

In [4]:
df.totalcharges

0         29.85
1        1889.5
2        108.15
3       1840.75
4        151.65
         ...   
7038     1990.5
7039     7362.9
7040     346.45
7041      306.6
7042     6844.5
Name: totalcharges, Length: 7043, dtype: object

In [5]:
df.dtypes

customerid           object
gender               object
seniorcitizen         int64
partner              object
dependents           object
tenure                int64
phoneservice         object
multiplelines        object
internetservice      object
onlinesecurity       object
onlinebackup         object
deviceprotection     object
techsupport          object
streamingtv          object
streamingmovies      object
contract             object
paperlessbilling     object
paymentmethod        object
monthlycharges      float64
totalcharges         object
churn                object
dtype: object

In [6]:
df['totalcharges'] = pd.to_numeric(df['totalcharges'], errors='coerce')  # convert to a numeric type
df['totalcharges'] = df['totalcharges'].fillna(0)  # Fill NaN values with 0
df.totalcharges.isnull().sum()

np.int64(0)

In [7]:
null_tc = df[df['totalcharges'].isnull()][['customerid', 'totalcharges']]
null_tc

,customerid,totalcharges


In [8]:
df.churn.head()   # churn variable

0     no
1     no
2    yes
3     no
4    yes
Name: churn, dtype: object

In [9]:
(df.churn == 'yes').head()

0    False
1    False
2     True
3    False
4     True
Name: churn, dtype: bool

In [10]:
(df.churn == 'yes').astype(int).head()

0    0
1    0
2    1
3    0
4    1
Name: churn, dtype: int64

In [11]:
df.churn = (df.churn == 'yes').astype(int)  # Replacing churn with 0 or 1

#### 2.0 Setting Up Validation Framework

In [12]:
from sklearn.model_selection import train_test_split

In [13]:
df_tr_tes_val, df_test =train_test_split(df, test_size = 0.2, random_state = 1) # Splitting data into train & test sets

df_train, df_val = train_test_split(df_tr_tes_val, test_size = 0.25, random_state = 1)

In [14]:
len(df_train), len(df_test), len(df_val)

(4225, 1409, 1409)

In [15]:
df_train = df_train.reset_index(drop = True)
df_test = df_test.reset_index(drop = True)
df_val = df_val.reset_index(drop = True)

In [16]:
y_train = df_train.churn.values
y_test = df_test.churn.values
y_val  = df_val.churn.values

In [17]:
del df_train['churn']  # Deleting churn variables from dataframe
del df_test['churn']
del df_val['churn']

#### 3.0 EDA

##### Checking for missing values

In [18]:
df_tr_tes_val = df_tr_tes_val.reset_index(drop= True)
df_tr_tes_val.isnull().sum()  # Checking for missing values

customerid          0
gender              0
seniorcitizen       0
partner             0
dependents          0
tenure              0
phoneservice        0
multiplelines       0
internetservice     0
onlinesecurity      0
onlinebackup        0
deviceprotection    0
techsupport         0
streamingtv         0
streamingmovies     0
contract            0
paperlessbilling    0
paymentmethod       0
monthlycharges      0
totalcharges        0
churn               0
dtype: int64

In [19]:
df_tr_tes_val.churn.value_counts(normalize = True)  # Churn rate

churn
0    0.730032
1    0.269968
Name: proportion, dtype: float64

In [20]:
df_tr_tes_val.dtypes

customerid           object
gender               object
seniorcitizen         int64
partner              object
dependents           object
tenure                int64
phoneservice         object
multiplelines        object
internetservice      object
onlinesecurity       object
onlinebackup         object
deviceprotection     object
techsupport          object
streamingtv          object
streamingmovies      object
contract             object
paperlessbilling     object
paymentmethod        object
monthlycharges      float64
totalcharges        float64
churn                 int64
dtype: object

##### Checking for numerical & categorical variables

In [21]:
df_tr_tes_val.dtypes
numerical = ['tenure','monthlycharges', 'totalcharges']

In [22]:
df_tr_tes_val.columns

Index(['customerid', 'gender', 'seniorcitizen', 'partner', 'dependents',
       'tenure', 'phoneservice', 'multiplelines', 'internetservice',
       'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport',
       'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling',
       'paymentmethod', 'monthlycharges', 'totalcharges', 'churn'],
      dtype='object')

In [23]:
categorical = ['customerid', 'gender', 'seniorcitizen', 'partner', 'dependents',
        'phoneservice', 'multiplelines', 'internetservice',
       'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport',
       'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling',
       'paymentmethod']

In [24]:
df_tr_tes_val[categorical].head()

,customerid,gender,seniorcitizen,partner,dependents,phoneservice,multiplelines,internetservice,onlinesecurity,onlinebackup,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod
0,5442-pptjy,male,0,yes,yes,yes,no,no,no_internet_service,no_internet_service,no_internet_service,no_internet_service,no_internet_service,no_internet_service,two_year,no,mailed_check
1,6261-rcvns,female,0,no,no,yes,no,dsl,yes,yes,yes,yes,no,yes,one_year,no,credit_card_(automatic)
2,2176-osjuv,male,0,yes,no,yes,yes,dsl,yes,yes,no,yes,no,no,two_year,no,bank_transfer_(automatic)
3,6161-erdgd,male,0,yes,yes,yes,yes,dsl,yes,no,yes,yes,yes,yes,one_year,no,electronic_check
4,2364-ufrom,male,0,no,no,yes,no,dsl,yes,yes,no,yes,yes,no,one_year,no,electronic_check


#### 4.0 Feature Importance

In [25]:
df_tr_tes_val.head()

,customerid,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,...,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn
0,5442-pptjy,male,0,yes,yes,12,yes,no,no,no_internet_service,...,no_internet_service,no_internet_service,no_internet_service,no_internet_service,two_year,no,mailed_check,19.70,258.35,0
1,6261-rcvns,female,0,no,no,42,yes,no,dsl,yes,...,yes,yes,no,yes,one_year,no,credit_card_(automatic),73.90,3160.55,1
2,2176-osjuv,male,0,yes,no,71,yes,yes,dsl,yes,...,no,yes,no,no,two_year,no,bank_transfer_(automatic),65.15,4681.75,0
3,6161-erdgd,male,0,yes,yes,71,yes,yes,dsl,yes,...,yes,yes,yes,yes,one_year,no,electronic_check,85.45,6300.85,0
4,2364-ufrom,male,0,no,no,30,yes,no,dsl,yes,...,no,yes,yes,no,one_year,no,electronic_check,70.40,2044.75,0


##### 4.1 Churn rate

In [26]:
male_churn = df_tr_tes_val[df_tr_tes_val.gender == 'male'].churn.mean()
male_churn

np.float64(0.2632135306553911)

In [27]:
female_churn = df_tr_tes_val[df_tr_tes_val.gender == 'female'].churn.mean()
female_churn

np.float64(0.27682403433476394)

In [28]:
global_churn = df_tr_tes_val.churn.mean()
global_churn

np.float64(0.26996805111821087)

In [29]:
df_tr_tes_val.partner.value_counts()

partner
no     2932
yes    2702
Name: count, dtype: int64

In [30]:
with_partner = df_tr_tes_val[df_tr_tes_val.partner == 'yes'].churn.mean()
with_partner

np.float64(0.20503330866025166)

In [31]:
with_no_partner = df_tr_tes_val[df_tr_tes_val.partner == 'no'].churn.mean()
with_no_partner

np.float64(0.3298090040927694)

In [32]:
global_churn - df_tr_tes_val[df_tr_tes_val.gender == 'male'].churn.mean()

np.float64(0.006754520462819769)

In [33]:
global_churn - with_partner

np.float64(0.06493474245795922)

In [34]:
global_churn - with_no_partner

np.float64(-0.05984095297455855)

##### It is observed that the partner variable is more important in predicting the churn rate compared with the gender variable.
##### Individuals with no partner are more likely to churn compared with those with partner.

##### 4.2 Risk ratio
#####     group_churn_rate/global_churn_rate  {if >1}, more likely to churn and vice-versa

In [35]:
with_partner/global_churn

np.float64(0.7594724924338315)

In [36]:
with_no_partner/global_churn

np.float64(1.2216593879412643)

In [37]:
male_churn/global_churn

np.float64(0.9749802969838747)

In [38]:
female_churn/global_churn

np.float64(1.0253955354648652)

##### SQL implementation 

In [39]:
for i in categorical:
    print(i)
    df_group = df_tr_tes_val.groupby('gender').churn.agg(['mean', 'count'])
    df_group['diff']= df_group['mean'] - global_churn
    df_group['risk']= df_group['mean']/global_churn
    display(df_group)
    print()
    print()

customerid


,mean,count,diff,risk
gender,,,,
female,0.276824,2796,0.006856,1.025396
male,0.263214,2838,-0.006755,0.974980




gender


,mean,count,diff,risk
gender,,,,
female,0.276824,2796,0.006856,1.025396
male,0.263214,2838,-0.006755,0.974980




seniorcitizen


...

,mean,count,diff,risk
gender,,,,
female,0.276824,2796,0.006856,1.025396
male,0.263214,2838,-0.006755,0.974980




paperlessbilling


,mean,count,diff,risk
gender,,,,
female,0.276824,2796,0.006856,1.025396
male,0.263214,2838,-0.006755,0.974980




paymentmethod


,mean,count,diff,risk
gender,,,,
female,0.276824,2796,0.006856,1.025396
male,0.263214,2838,-0.006755,0.974980


##### 4.3 Feature inportance: mutual information
######    Mutual information tells us how much we can learn about a variable if we know the value of another.

In [40]:
from sklearn.metrics import mutual_info_score

In [41]:
mutual_info_score(df_tr_tes_val.contract, df_tr_tes_val.churn)

np.float64(0.0983203874041556)

In [42]:
mutual_info_score(df_tr_tes_val.gender, df_tr_tes_val.churn)

np.float64(0.0001174846211139946)

In [43]:
mutual_info_score(df_tr_tes_val.partner, df_tr_tes_val.churn)

np.float64(0.009967689095399745)

In [44]:
def mutual_info_churn_score(series):
    return mutual_info_score(series, df_tr_tes_val.churn)

In [45]:
churn_score = df_tr_tes_val[categorical].apply(mutual_info_churn_score)
churn_score.sort_values(ascending = False)

customerid          0.583227
contract            0.098320
onlinesecurity      0.063085
techsupport         0.061032
internetservice     0.055868
onlinebackup        0.046923
deviceprotection    0.043453
paymentmethod       0.043210
streamingtv         0.031853
streamingmovies     0.031581
paperlessbilling    0.017589
dependents          0.012346
partner             0.009968
seniorcitizen       0.009410
multiplelines       0.000857
phoneservice        0.000229
gender              0.000117
dtype: float64

###### Customers with the highest churn rates are of more importance

##### 4.4 Correlation
###### It tells us the degree of dependance between two variables

In [46]:
df_tr_tes_val[numerical].corrwith(df_tr_tes_val.churn)

tenure           -0.351885
monthlycharges    0.196805
totalcharges     -0.196353
dtype: float64

#### 5.0 One-Hot Encoding

In [47]:
from sklearn.feature_extraction import DictVectorizer

In [48]:
train_dict = df_train[categorical + numerical].to_dict(orient='records')
train_dict[0]

{'customerid': '8015-ihcgw',
 'gender': 'female',
 'seniorcitizen': 0,
 'partner': 'yes',
 'dependents': 'yes',
 'phoneservice': 'yes',
 'multiplelines': 'yes',
 'internetservice': 'fiber_optic',
 'onlinesecurity': 'yes',
 'onlinebackup': 'yes',
 'deviceprotection': 'yes',
 'techsupport': 'yes',
 'streamingtv': 'yes',
 'streamingmovies': 'yes',
 'contract': 'two_year',
 'paperlessbilling': 'yes',
 'paymentmethod': 'electronic_check',
 'tenure': 72,
 'monthlycharges': 115.5,
 'totalcharges': 8425.15}

In [49]:
dv = DictVectorizer(sparse = False)

In [50]:
dv.fit(train_dict)
X_train = dv.fit_transform(train_dict)

In [51]:
val_dict = df_val[categorical + numerical].to_dict(orient='records')

In [52]:
X_val = dv.transform(val_dict)

#### 6.0 Logistic Regression

In [53]:
def linear_regression(ki):
    result = w0

    for i in range(len(w)):
        result = result + ki[i]*w[i]
    
    return result

In [54]:
def logistic_regression(ki):
    score = w0

    for i in range(len(w)):
        score = score + ki[i]*w[i]

    result = sigmoid(score)
    return result

##### 6.1 Train model with scikit-learn

In [55]:
from sklearn.linear_model import LogisticRegression

In [56]:
model = LogisticRegression()
model.fit(X_train, y_train)

/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [57]:
model.intercept_ # bias term

array([-0.107759])

In [58]:
model.coef_  # weights

array([[ 5.31109947e-01, -1.83252777e-01, -4.54026662e-01, ...,
        -2.32078946e-01, -7.13843302e-02,  3.88423719e-04]])

In [59]:
y_pred = model.predict_proba(X_val)[:,1] # probability of churning

In [60]:
churn_decision = y_pred >= 0.5

In [61]:
df_val[churn_decision].customerid

3       8433-wxgna
8       3440-jpscl
11      2637-fkfsy
12      7228-omtpn
19      6711-fldfb
           ...    
1397    5976-jcjrh
1398    2034-cgrhz
1399    5276-kqwhg
1407    6521-yytyi
1408    3049-solay
Name: customerid, Length: 315, dtype: object

In [62]:
y_val

array([0, 0, 0, ..., 0, 1, 1])

In [63]:
churn_decision.astype(int)

array([0, 0, 0, ..., 0, 1, 1])

In [64]:
(y_val == churn_decision).mean() # checking model correctness

np.float64(0.8005677785663591)

#### 7.0 Model Interpretation 

##### 7.1 Model coefficients

In [65]:
features_weight = dict(zip(dv.get_feature_names_out(), model.coef_[0])) # checking the weight of each variable

In [66]:
# Training a smaller model taking a subset of features

sub = ['contract','tenure','monthlycharges']
df_train[sub]

,contract,tenure,monthlycharges
0,two_year,72,115.50
1,month-to-month,10,95.25
2,month-to-month,5,75.55
3,month-to-month,5,80.85
4,two_year,18,20.10
...,...,...,...
4220,one_year,52,80.85
4221,month-to-month,18,25.15
4222,month-to-month,2,90.00
4223,two_year,27,24.50


In [67]:
train_sub = df_train[sub].to_dict(orient='records')
val_sub = df_val[sub].to_dict(orient='records')

In [68]:
dv_sub = DictVectorizer(sparse = False)

dv_sub.fit(train_sub)
#dv_sub_model.fit(val_sub_model)

DictVectorizer(sparse=False)

In [69]:
dv_sub.get_feature_names_out()

array(['contract=month-to-month', 'contract=one_year',
       'contract=two_year', 'monthlycharges', 'tenure'], dtype=object)

In [70]:
X_train_sub = dv_sub.transform(train_sub)

In [71]:
sub_model = LogisticRegression()
sub_model.fit(X_train_sub, y_train)

LogisticRegression()

In [72]:
sub_model.intercept_[0] # bias term

np.float64(-2.477957595766546)

In [73]:
sub_model.coef_[0] # weight

array([ 0.9711394 , -0.02379507, -0.94828863,  0.02748534, -0.03619005])

In [74]:
dict(zip(dv.get_feature_names_out(), sub_model.coef_[0].round(3))) # checking the weight of each variable

{'contract=month-to-month': np.float64(0.971),
 'contract=one_year': np.float64(-0.024),
 'contract=two_year': np.float64(-0.948),
 'customerid=0002-orfbo': np.float64(0.027),
 'customerid=0011-igkff': np.float64(-0.036)}

##### It can be inferred from above that individuals with monthly contracts have high churn rate compared with other contracts

#### 8.0 Using the model

In [75]:
dict_df_tr_tes_val = df_tr_tes_val[categorical + numerical].to_dict(orient='records')

In [76]:
dv = DictVectorizer(sparse = False)
X_df_tr_tes_val = dv.fit_transform(dict_df_tr_tes_val)

In [77]:
y_df_tr_tes_val = df_tr_tes_val.churn.values

In [78]:
model = LogisticRegression()
model.fit(X_df_tr_tes_val, y_df_tr_tes_val)

/home/codespace/.local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [79]:
dict_test = df_test[categorical + numerical].to_dict(orient='records')

In [80]:
X_test = dv.transform(dict_test)

In [81]:
y_pred = model.predict_proba(X_test)[:,1]

In [82]:
churn_decision = (y_pred >= 0.5)

In [83]:
(churn_decision == y_test).mean()

np.float64(0.8119233498935415)

In [84]:
df_test[churn_decision].customerid

10      0111-klbqg
12      6168-wfvvf
30      6402-zfppi
35      0362-zbzwj
37      4559-uwiht
           ...    
1392    5134-ikday
1395    4910-aqffx
1403    2215-zafgx
1404    5130-iekqt
1408    9874-qlclh
Name: customerid, Length: 327, dtype: object

In [85]:
customer = dict_test[10]
customer

{'customerid': '0111-klbqg',
 'gender': 'male',
 'seniorcitizen': 1,
 'partner': 'yes',
 'dependents': 'yes',
 'phoneservice': 'yes',
 'multiplelines': 'no',
 'internetservice': 'fiber_optic',
 'onlinesecurity': 'no',
 'onlinebackup': 'yes',
 'deviceprotection': 'no',
 'techsupport': 'no',
 'streamingtv': 'yes',
 'streamingmovies': 'yes',
 'contract': 'month-to-month',
 'paperlessbilling': 'yes',
 'paymentmethod': 'mailed_check',
 'tenure': 32,
 'monthlycharges': 93.95,
 'totalcharges': 2861.45}

In [86]:
 customer_1 = dv.transform([customer])

In [87]:
customer_1.shape

(1, 5679)

In [88]:
model.predict_proba(customer_1)[0,1]

np.float64(0.5216310500098315)

In [89]:
y_test[10]

np.int64(0)